# Agent 流式输出模式进阶

上一节列出了 `values`、`updates`、`messages`、`tasks`、`debug`、`checkpoints`、`custom` 七种模式。本节回答两个更实际的问题：

1. **每一种模式的 Chunk 到底长什么样，携带的是 token、状态，还是任务事件？**
2. **只选择 `custom` 时，模型回复和模型 reasoning 会不会消失？**

先给结论：`stream_mode` 是“订阅哪些流”的选择，不是对 Agent 能力的开关。只订阅 `custom`，就只会收到工具或节点主动写出的自定义事件；模型仍然会执行并生成回复，只是这些回复没有被输出到当前迭代器。需要同时展示模型文本、工具进度和执行步骤时，传入多个模式，例如：

```python
stream_mode=["messages", "updates", "custom"]
```

本笔记统一使用 `version="v2"`。在当前 LangGraph 中，它让所有模式都使用相同的外层结构，避免旧版 v1 在单模式、多模式时返回值外形不同的问题。


## 1. Chunk 的统一外层：先看 v2

`version="v2"` 时，每一个 Chunk 都是一个 `StreamPart` 字典：

```python
{
    "type": "updates" | "messages" | "custom" | ...,
    "ns": (),       # 命名空间；有子图/子 Agent 时用于标识来源
    "data": ...,    # 真正的载荷，随模式而变化
}
```

| 模式 | `data` 的典型形态 | 它回答的问题 |
| --- | --- | --- |
| `values` | **完整 State**，如 `{"messages": [...]}` | 当前图执行完这一步后的完整状态是什么？ |
| `updates` | `{节点名: 状态增量}`，如 `{"model": {"messages": [AIMessage(...)]}}` | 刚刚是哪个节点改了什么？ |
| `messages` | `(message_chunk, metadata)` | LLM 此刻吐出了哪个 token、工具调用参数片段或 reasoning 片段？来自哪个节点？ |
| `custom` | `writer(...)` 写入的任意 Python 数据 | 工具/节点想主动报告什么业务进度？ |
| `tasks` | task 的启动、结束、结果或错误事件 | 图中任务的生命周期如何？ |
| `debug` | 更完整的调试事件（任务、状态、时间等） | 图为什么按这个顺序执行？ |
| `checkpoints` | 检查点事件，形态接近 `get_state()` 的结果 | 可恢复会话在何处、以什么状态被持久化？ |

注意两点：

- `values` 与 `updates` 的核心区别不是“是否流式”，而是**完整状态快照**与**增量状态变化**的区别。
- `messages` 中不只有最后的自然语言文本。一次工具调用会先出现 `tool_call_chunk`（工具名和 JSON 参数逐步生成），工具执行完成后还会出现工具消息，最后才是模型的最终回答 token。


## 2. v1 与 v2：为什么建议显式写 `version="v2"`

原笔记没有传 `version`，因此走的是兼容旧代码的 v1 格式：

| 场景 | v1（默认） | v2（推荐） |
| --- | --- | --- |
| 单一模式 | 直接得到裸 `data` | `{"type", "ns", "data"}` |
| 多个模式 | `(mode, data)` 元组 | 仍是统一的 `StreamPart` 字典 |
| 含子图/子 Agent | 元组层级还会继续增加 | 通过 `chunk["ns"]` 判断来源 |

因此本节后续代码总是通过 `chunk["type"]` 分发，而不是靠“这个 Chunk 是元组还是字典”来猜测。你的环境中 `langgraph` 已满足 v2 所需的版本。


In [ ]:
# 公共准备：一个会发出 custom 进度的工具，以及带检查点的 Agent。
# 后面的所有示例都复用它；请先运行本单元格。

import os
import time

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)


@tool
def generate_sales_report() -> str:
    """生成销售报告。处理报告请求时必须调用此工具。"""
    writer = get_stream_writer()

    # writer(...) 不会改变工具的返回值；它只额外向 custom 流发送事件。
    writer({"event": "progress", "step": "load_data", "progress": 0})
    time.sleep(0.3)  # 模拟工具在等待/处理数据，便于观察进度事件的先后顺序。

    writer({"event": "progress", "step": "aggregate", "progress": 50})
    time.sleep(0.3)

    writer({"event": "progress", "step": "complete", "progress": 100})
    return "销售报告：本月营收 150 万元，环比增长 12%。"


agent = create_agent(
    model=model,
    tools=[generate_sales_report],
    checkpointer=InMemorySaver(),  # checkpoints 模式必须有 checkpointer。
    system_prompt="用户要求生成销售报告时，必须调用 generate_sales_report 工具，再根据工具结果回答。",
)

inputs = {"messages": [{"role": "user", "content": "请生成一份销售报告"}]}
config = {"configurable": {"thread_id": "streaming-advanced-demo"}}


In [ ]:
# 观察单个模式的原始 Chunk。
# 改变 STREAM_MODE 后重新运行即可比较七种模式；不要同时选多个，
# 否则会混合不同类型的事件，不利于先建立“单种 Chunk”的直觉。

STREAM_MODE = "updates"
# 可选值："values"、"updates"、"messages"、"custom"、
#         "tasks"、"debug"、"checkpoints"

for chunk in agent.stream(
    inputs,
    config=config,
    stream_mode=STREAM_MODE,
    version="v2",
):
    print(f"type = {chunk['type']}")
    print(f"namespace = {chunk['ns']}")
    rprint(chunk["data"])
    print("-" * 70)

# 观察建议：
# - values：每次都能看到完整的 messages 列表，信息最多、重复也最多。
# - updates：通常依次看到 model -> tools -> model 三个节点的变化。
# - messages：data 是 (message_chunk, metadata)，会细到 token/工具参数片段。
# - custom：只能看到本工具中 writer(...) 写入的三个进度字典。
# - checkpoints：依赖上个单元格中的 InMemorySaver。


## 3. `custom` 不会让模型“停止输出”，但 custom-only 看不到模型流

`get_stream_writer()` 的职责很窄：它允许工具或任意 LangGraph 节点主动向 **custom 流**塞入数据。它不会自动把模型文本、工具返回值或思考过程复制到 custom 流。

所以原笔记的这段代码：

```python
agent.stream(..., stream_mode="custom")
```

只能看到 `writer(...)` 写出的进度。这不是模型回复被隐藏或丢失，而是当前消费者只订阅了 custom 这个频道。解决方式是组合模式：

```python
stream_mode=["messages", "updates", "custom"]
```

- `messages`：实时模型输出，包含文本 token、逐步生成的工具调用参数；如果模型和供应商开放 reasoning，也会以标准 `reasoning` content block 出现。
- `updates`：完整的工具调用/工具结果/最终 `AIMessage`，适合获得已解析的工具参数和最终消息。
- `custom`：业务进度，例如“已读取 50% 数据”。

关于“模型思考”：只能展示模型 API 实际返回、且已启用的 reasoning 内容。很多模型不会返回原始内部思维链，或只返回摘要；没有返回的内容无法通过 `custom` 推导出来。`messages` 是消费这些**可用 reasoning block**的正确渠道。


In [ ]:
# 推荐的实战写法：同时消费模型流、状态更新与自定义进度。
# 这正是 custom 与模型回复并存的方式。

from langchain_core.messages import AIMessageChunk


def print_message_part(message_chunk, metadata):
    """按 content block 类型分别处理文本、reasoning 与工具调用增量。"""
    node_name = metadata.get("langgraph_node", "unknown")

    if isinstance(message_chunk, AIMessageChunk):
        for block in message_chunk.content_blocks:
            block_type = block.get("type")

            if block_type == "text":
                # 最终回答的打字机效果通常来自这里。
                print(block.get("text", ""), end="", flush=True)
            elif block_type == "reasoning":
                # 只有供应商实际提供且模型已启用 reasoning 时才会进入这里。
                print(f"[reasoning] {block.get('reasoning', '')}", end="", flush=True)
            elif block_type == "tool_call_chunk":
                # 工具名/参数是流式生成的，单个 chunk 往往只是 JSON 的一部分。
                print(
                    f"\n[tool_call_chunk] name={block.get('name')} args={block.get('args')}",
                    flush=True,
                )
    elif message_chunk.content:
        # 例如 tools 节点返回的 ToolMessage。
        print(f"\n[tool result from {node_name}] {message_chunk.content}")


for chunk in agent.stream(
    inputs,
    config=config,
    stream_mode=["messages", "updates", "custom"],
    version="v2",
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        print_message_part(message_chunk, metadata)

    elif chunk["type"] == "custom":
        # 这是工具通过 writer(...) 主动发出的业务事件。
        print(f"\n[progress] {chunk['data']}")

    elif chunk["type"] == "updates":
        # updates 是完成一个节点后的状态增量，不是 token。
        print(f"\n[state update] nodes={list(chunk['data'])}")


## 4. 选择建议与官方文档变化

| 你的目标 | 常用组合 |
| --- | --- |
| 聊天界面的打字机回答 | `messages` |
| 同时展示“正在调用什么工具”与最终文本 | `messages + updates` |
| 同时展示工具进度条、模型回答与工具状态 | `messages + updates + custom` |
| 排查图执行顺序或子任务问题 | `debug` 或 `tasks` |
| 观察/恢复会话状态 | `checkpoints`，并配置 checkpointer |
| 只关心最终每步状态 | `values` |

LangChain 当前的 Agent 流式文档重点讲解 `updates`、`messages`、`custom` 三个最常用频道；更底层的 LangGraph 流式文档补充 `values`、`tasks`、`debug`、`checkpoints`。对新应用，LangChain 还推荐了解更高层的 Event Streaming：它为 messages、state、tool calls 等提供分离的类型化迭代器。不过学习 Agent 和处理自定义节点进度时，`stream_mode` 仍然是最直接、最值得掌握的基础。

参考：

- [LangChain Agent Streaming](https://docs.langchain.com/oss/python/langchain/streaming)
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)
- [LangChain Event Streaming](https://docs.langchain.com/oss/python/langchain/event-streaming)
